# 04 — Predicción de No-Shows en reservar

**Objetivo:** xxx.

El propósito de este notebook es xxx:

- xxx
- xxx

xxx.

*Arquitectura Medallion: Bronze → Silver → **Gold***

### 0. Setup

Imports, rutas, carga de los parquets de Silver. Una sola celda de configuración compartida.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Rutas del proyecto ────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

DATA_DIR    = PROJECT_ROOT / 'data'
BRONZE_SNAP = DATA_DIR / 'bronze' / 'snapshots'
SILVER_SNAP = DATA_DIR / 'silver' / 'snapshots'
GOLD_DIR    = DATA_DIR / 'gold'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures' / 'eda'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Bronze snaps : {BRONZE_SNAP}')
print(f'Silver snaps : {SILVER_SNAP}')
print(f'Gold dir     : {GOLD_DIR}')
print(f'Figures dir  : {FIGURES_DIR}')

Project root : /Users/laura/TFM-Hosteleria-AI
Bronze snaps : /Users/laura/TFM-Hosteleria-AI/data/bronze/snapshots
Silver snaps : /Users/laura/TFM-Hosteleria-AI/data/silver/snapshots
Gold dir     : /Users/laura/TFM-Hosteleria-AI/data/gold
Figures dir  : /Users/laura/TFM-Hosteleria-AI/results/figures/eda


In [2]:
df_reservas = pd.read_parquet(SILVER_SNAP / 'reservas_silver.parquet')
master = pd.read_parquet(GOLD_DIR / 'tabla_maestra_diaria.parquet')

In [3]:
df_reservas["status"].value_counts()

status
Sentada                         17080
Cancelado por el cliente         2875
Liberada                          662
No show                           493
Confirmada                        135
Cuenta solicitada                  13
Llegada                            13
Cancelado por el restaurante       11
A revisar                           3
Pendiente                           2
Reconfirmada                        2
Name: count, dtype: int64

In [4]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# 1. ¿'Confirmada' / 'Reconfirmada' son reservas aún pendientes (recientes)
#    o reservas realmente completadas (repartidas por todo el histórico)?
# ---------------------------------------------------------------------------

fecha_max_dataset = df_reservas['reservation_date'].max()
fecha_min_dataset = df_reservas['reservation_date'].min()
print(f"Rango del dataset completo: {fecha_min_dataset} → {fecha_max_dataset}\n")

estados_dudosos_confirmacion = ['Confirmada', 'Reconfirmada']
subset_confirmacion = df_reservas[df_reservas['status'].isin(estados_dudosos_confirmacion)]

dias_desde_max_confirmacion = (fecha_max_dataset - subset_confirmacion['reservation_date']).dt.days
print("Antigüedad en días respecto al máximo del dataset — Confirmada/Reconfirmada:")
print(dias_desde_max_confirmacion.describe())
print()

# Grupo de referencia: 'Sentada', que sabemos con certeza que es un desenlace resuelto.
subset_sentada = df_reservas[df_reservas['status'] == 'Sentada']
dias_desde_max_sentada = (fecha_max_dataset - subset_sentada['reservation_date']).dt.days
print("Antigüedad en días respecto al máximo del dataset — Sentada (referencia):")
print(dias_desde_max_sentada.describe())
print()

print("Interpretación: si la mediana de 'Confirmada/Reconfirmada' es mucho menor")
print("que la de 'Sentada' (están concentradas cerca del final del dataset),")
print("son reservas todavía no resueltas -> tratar como 'pendiente'.")
print("Si las medianas son parecidas, están repartidas como cualquier otra")
print("reserva resuelta -> más defendible tratarlas como 'completada'.\n")


# ---------------------------------------------------------------------------
# 2. ¿'Liberada' se parece más a una reserva completada o a una cancelada?
#    Comparación de la tasa de asignación de mesa (columna 'table')
# ---------------------------------------------------------------------------

estados_referencia = ['Sentada', 'Cancelado por el cliente', 'No show', 'Liberada']
subset_ref = df_reservas[df_reservas['status'].isin(estados_referencia)].copy()

def es_mesa_valida(valor):
    """
    La columna 'table' tiene valores corruptos conocidos (fechas mal parseadas
    que deberían ser números de mesa, detectado en la auditoría de Silver).
    Filtramos esos valores para no contaminar el diagnóstico.
    """
    if pd.isna(valor):
        return False
    texto = str(valor)
    if '-' in texto or ':' in texto:  # patrón típico de fecha/hora mal parseada
        return False
    return True

subset_ref['mesa_valida'] = subset_ref['table'].apply(es_mesa_valida)

tasa_asignacion_mesa = (
    subset_ref
    .groupby('status')['mesa_valida']
    .mean()
    .sort_values(ascending=False)
)
print("Tasa de asignación de mesa válida por estado:")
print(tasa_asignacion_mesa)
print()
print("Interpretación: si 'Liberada' tiene una tasa parecida a 'Sentada',")
print("apunta a servicio completado. Si se parece a 'Cancelado por el cliente'")
print("o 'No show', apunta a cancelación.\n")


# ---------------------------------------------------------------------------
# 3. Perfil de reservas (people, shift, origin) de 'Liberada' 
#    comparado con las categorías ya conocidas
# ---------------------------------------------------------------------------

print("Estadísticos de 'people' (tamaño de grupo) por estado:")
print(subset_ref.groupby('status')['people'].describe())
print()

print("Distribución de 'shift' por estado (%):")
print((pd.crosstab(subset_ref['status'], subset_ref['shift'], normalize='index') * 100).round(1))
print()

print("Distribución de 'origin' por estado (%):")
print((pd.crosstab(subset_ref['status'], subset_ref['origin'], normalize='index') * 100).round(1))
print()

print("Interpretación: cuanto más se parezca el perfil de 'Liberada' al de")
print("'Sentada', más plausible es tratarla como completada. Cuanto más se")
print("parezca a 'Cancelado por el cliente' o 'No show', más plausible es")
print("tratarla como cancelada/no-show.")

Rango del dataset completo: 2022-11-10 00:00:00 → 2026-07-08 00:00:00

Antigüedad en días respecto al máximo del dataset — Confirmada/Reconfirmada:
count     137.000000
mean      693.569343
std       389.061272
min        32.000000
25%       407.000000
50%       587.000000
75%       992.000000
max      1329.000000
Name: reservation_date, dtype: float64

Antigüedad en días respecto al máximo del dataset — Sentada (referencia):
count    17080.000000
mean       585.953279
std        396.665591
min          0.000000
25%        235.000000
50%        515.000000
75%        932.000000
max       1329.000000
Name: reservation_date, dtype: float64

Interpretación: si la mediana de 'Confirmada/Reconfirmada' es mucho menor
que la de 'Sentada' (están concentradas cerca del final del dataset),
son reservas todavía no resueltas -> tratar como 'pendiente'.
Si las medianas son parecidas, están repartidas como cualquier otra
reserva resuelta -> más defendible tratarlas como 'completada'.

Tasa de asignac

In [5]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# 0. Variable de antelación (lead time) — la vamos a necesitar como feature
#    más adelante, así que la calculamos ya de forma limpia y reutilizable
# ---------------------------------------------------------------------------

df_reservas['antelacion_horas'] = (
    (df_reservas['reservation_datetime'] - df_reservas['created_datetime'])
    .dt.total_seconds() / 3600
)

# Chequeo de sanidad: no debería haber antelación negativa (reservado
# después de la hora del servicio). Si aparecen, son un problema de
# calidad de datos a resolver en Silver, no algo que arrastrar al modelo.
n_negativas = (df_reservas['antelacion_horas'] < 0).sum()
print(f"Filas con antelación negativa (revisar si aparecen): {n_negativas}\n")


# ---------------------------------------------------------------------------
# 1. Distribución de antelación por canal (origin) — visión general
# ---------------------------------------------------------------------------

print("Antelación en horas, por origin (describe):")
print(df_reservas.groupby('origin')['antelacion_horas'].describe())
print()


# ---------------------------------------------------------------------------
# 2. Foco en walk-in: ¿es ocupación inmediata (antelación ~0) o reserva
#    presencial para cualquier fecha (antelación con recorrido real)?
# ---------------------------------------------------------------------------

subset_walkin = df_reservas[df_reservas['origin'] == 'walk in']

print(f"Total reservas walk-in: {len(subset_walkin)}\n")
print("Antelación en horas, dentro de walk-in (describe):")
print(subset_walkin['antelacion_horas'].describe())
print()

# % de walk-in con antelación por debajo de distintos umbrales.
# Si la inmensa mayoría cae en "menos de 1h", apoya la interpretación
# clásica (ocupación inmediata). Si se reparte también en umbrales
# grandes, apoya tu interpretación (canal presencial, fecha cualquiera).
umbrales_horas = [1, 6, 24, 24 * 7]  # 1h, 6h, 1 día, 1 semana
for umbral in umbrales_horas:
    pct = (subset_walkin['antelacion_horas'] < umbral).mean() * 100
    print(f"% walk-in con antelación < {umbral}h: {pct:.1f}%")
print()

# ---------------------------------------------------------------------------
# 3. Comprobación de bimodalidad dentro de walk-in
#    (¿hay dos poblaciones distintas mezcladas bajo la misma etiqueta?)
# ---------------------------------------------------------------------------

bins_antelacion = [-np.inf, 1, 6, 24, 24 * 7, 24 * 30, np.inf]
labels_antelacion = ['<1h', '1-6h', '6-24h', '1-7 días', '7-30 días', '>30 días']

subset_walkin = subset_walkin.copy()
subset_walkin['tramo_antelacion'] = pd.cut(
    subset_walkin['antelacion_horas'],
    bins=bins_antelacion,
    labels=labels_antelacion,
)

print("Distribución de walk-in por tramo de antelación (conteo | %):")
conteo_tramo = subset_walkin['tramo_antelacion'].value_counts().sort_index()
pct_tramo = (subset_walkin['tramo_antelacion'].value_counts(normalize=True).sort_index() * 100).round(1)
print(pd.concat([conteo_tramo, pct_tramo], axis=1, keys=['n', 'pct']))
print()
print("Interpretación: si casi todo cae en '<1h', walk-in = ocupación")
print("inmediata (interpretación clásica). Si hay masa relevante repartida")
print("en tramos largos también, walk-in mezcla dos comportamientos")
print("distintos bajo la misma etiqueta -> considerar una variable")
print("'es_reserva_inmediata' aparte, en vez de depender solo de origin.\n")


# ---------------------------------------------------------------------------
# 4. Código pendiente de la comprobación anterior: status dentro de
#    walk-in vs no walk-in (interpretar ya con el contexto del punto 2-3)
# ---------------------------------------------------------------------------

conteo_walkin = subset_walkin['status'].value_counts()
pct_walkin = (subset_walkin['status'].value_counts(normalize=True) * 100).round(1)
print("Status dentro de walk-in (conteo | %):")
print(pd.concat([conteo_walkin, pct_walkin], axis=1, keys=['n', 'pct']))

subset_no_walkin = df_reservas[df_reservas['origin'] != 'walk in']
conteo_no_walkin = subset_no_walkin['status'].value_counts()
pct_no_walkin = (subset_no_walkin['status'].value_counts(normalize=True) * 100).round(1)
print("\nStatus dentro de NO walk-in (conteo | %):")
print(pd.concat([conteo_no_walkin, pct_no_walkin], axis=1, keys=['n', 'pct']))

Filas con antelación negativa (revisar si aparecen): 3096

Antelación en horas, por origin (describe):
            count       mean         std        min       25%        50%  \
origin                                                                     
appmovil    204.0  91.765377  189.389278  -4.329722  2.637708  24.723472   
moduloweb  5255.0  63.022266  140.268990   0.252222  3.804444  22.378056   
software   6421.0  35.693950  133.485791  -7.107222  1.153333   3.780000   
terceros   6346.0  43.166772   81.270760   0.250556  2.782639  16.700417   
walk in    3063.0  -0.150906    0.408853 -22.193889 -0.205833  -0.143056   

                 75%          max  
origin                             
appmovil   73.657847  1322.753056  
moduloweb  60.709028  2186.202222  
software   25.491944  5020.796667  
terceros   47.846389   721.046667  
walk in    -0.075556     0.823056  

Total reservas walk-in: 3063

Antelación en horas, dentro de walk-in (describe):
count    3063.000000
mean     

In [6]:
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Repetir la comparación de perfil, pero excluyendo walk-in de ambos lados
#    para eliminar la confusión que detectamos
# ---------------------------------------------------------------------------

estados_referencia = ['Sentada', 'Cancelado por el cliente', 'No show', 'Liberada']
subset_no_walkin_ref = df_reservas[
    (df_reservas['status'].isin(estados_referencia)) &
    (df_reservas['origin'] != 'walk in')
].copy()

print(f"Tamaño de cada grupo (excluyendo walk-in):")
print(subset_no_walkin_ref['status'].value_counts())
print()

print("Antelación en horas, por status (excluyendo walk-in):")
print(subset_no_walkin_ref.groupby('status')['antelacion_horas'].describe())
print()

print("Estadísticos de 'people', por status (excluyendo walk-in):")
print(subset_no_walkin_ref.groupby('status')['people'].describe())
print()

print("Distribución de 'shift', por status (excluyendo walk-in, %):")
print((pd.crosstab(subset_no_walkin_ref['status'], subset_no_walkin_ref['shift'], normalize='index') * 100).round(1))
print()

print("Distribución de 'origin' (dentro de los no walk-in), por status (%):")
print((pd.crosstab(subset_no_walkin_ref['status'], subset_no_walkin_ref['origin'], normalize='index') * 100).round(1))
print()


# ---------------------------------------------------------------------------
# 2. Distribución temporal de Liberada (no walk-in) vs el resto:
#    ¿es un concepto estable en el tiempo o concentrado en un periodo,
#    lo que apuntaría a un cambio de proceso/sistema?
# ---------------------------------------------------------------------------

fecha_max_dataset = df_reservas['reservation_date'].max()

for estado in estados_referencia:
    subset_estado = df_reservas[
        (df_reservas['status'] == estado) &
        (df_reservas['origin'] != 'walk in')
    ]
    antiguedad = (fecha_max_dataset - subset_estado['reservation_date']).dt.days
    print(f"Antigüedad en días (excl. walk-in) — {estado}:")
    print(antiguedad.describe())
    print()

print("Interpretación: si la distribución de antigüedad de 'Liberada' es")
print("parecida a la de 'Sentada' y 'Cancelado por el cliente' (repartida")
print("por todo el histórico), es un concepto estable. Si se concentra en")
print("un tramo concreto de fechas (muy distinto a las demás), sospecha de")
print("un cambio de proceso o de versión del sistema en ese periodo.\n")


# ---------------------------------------------------------------------------
# 3. Chequeo adicional barato: ¿'Liberada' se asocia con reservas de grupo?
# ---------------------------------------------------------------------------

print("Distribución de 'group' por status (excluyendo walk-in, %):")
print((pd.crosstab(subset_no_walkin_ref['status'], subset_no_walkin_ref['group'], normalize='index') * 100).round(1))

Tamaño de cada grupo (excluyendo walk-in):
status
Sentada                     14241
Cancelado por el cliente     2835
No show                       493
Liberada                      479
Name: count, dtype: int64

Antelación en horas, por status (excluyendo walk-in):
                            count       mean         std       min       25%  \
status                                                                         
Cancelado por el cliente   2835.0  79.477258  161.443034 -4.329722  5.355000   
Liberada                    479.0  28.363605   78.673904 -0.386944  1.449583   
No show                     493.0  68.591673  169.205004 -0.660556  2.156111   
Sentada                   14241.0  39.969088  109.387446 -2.595556  1.817222   

                                50%        75%          max  
status                                                       
Cancelado por el cliente  25.918611  74.529444  1944.328889  
Liberada                   5.585556  24.310417  1231.872500  
No s

In [7]:
import pandas as pd

# ---------------------------------------------------------------------------
# Chequeo temporal definitivo: ¿la proporción de 'Liberada' es estable a lo
# largo del histórico, o hay un tramo donde se dispara o desaparece? Esto es
# la prueba más directa de si es un concepto estable o un artefacto de un
# cambio de proceso/sistema de reservas.
# ---------------------------------------------------------------------------

df_no_walkin = df_reservas[df_reservas['origin'] != 'walk in'].copy()
df_no_walkin['periodo'] = df_no_walkin['reservation_date'].dt.to_period('Q')  # trimestral

tabla_periodo = (
    df_no_walkin
    .groupby(['periodo', 'status'])
    .size()
    .unstack(fill_value=0)
)

tabla_periodo['total'] = tabla_periodo.sum(axis=1)
tabla_periodo['n_liberada'] = tabla_periodo.get('Liberada', 0)
tabla_periodo['pct_liberada'] = (tabla_periodo['n_liberada'] / tabla_periodo['total'] * 100).round(2)

print("Evolución trimestral de 'Liberada' (excl. walk-in):")
print(tabla_periodo[['total', 'n_liberada', 'pct_liberada']])

Evolución trimestral de 'Liberada' (excl. walk-in):
status   total  n_liberada  pct_liberada
periodo                                 
2022Q4     721         146         20.25
2023Q1    1410          20          1.42
2023Q2    1278          18          1.41
2023Q3     944          15          1.59
2023Q4    1284          11          0.86
2024Q1    1310          13          0.99
2024Q2    1167           8          0.69
2024Q3     862           8          0.93
2024Q4    1295          17          1.31
2025Q1    1447          45          3.11
2025Q2    1270          50          3.94
2025Q3     926          18          1.94
2025Q4    1340          30          2.24
2026Q1    1427          36          2.52
2026Q2    1449          44          3.04
2026Q3      96           0          0.00


In [8]:
import pandas as pd

# ---------------------------------------------------------------------------
# Diagnóstico de la cola de walk-in: identificar cuántos casos hay más allá
# de lo que el patrón "minutos después de sentarse" explicaría, y mirarlos
# uno a uno para entender si es un problema puntual o sistemático.
# ---------------------------------------------------------------------------

subset_walkin = df_reservas[df_reservas['origin'] == 'walk in'].copy()

# Umbral de referencia: más de 2 horas de desfase ya no encaja con
# "el camarero lo introduce poco después de sentar al cliente".
UMBRAL_HORAS = -2

cola_walkin = subset_walkin[subset_walkin['antelacion_horas'] < UMBRAL_HORAS].copy()
print(f"Walk-in con antelación < {UMBRAL_HORAS}h: {len(cola_walkin)} de {len(subset_walkin)} "
      f"({len(cola_walkin) / len(subset_walkin) * 100:.2f}%)\n")

# Distribución completa de esa cola, para ver si son casos dispersos
# o se agrupan en un rango concreto (p.ej. todos entre -20h y -24h,
# lo que apuntaría a "un día de desfase" como patrón sistemático)
print("Distribución de antelación dentro de la cola:")
print(cola_walkin['antelacion_horas'].describe())
print()

# Casos concretos: fecha y hora de reserva vs fecha y hora de creación,
# para ver si el patrón es "mismo día distinto turno" o "día siguiente"
cola_walkin_detalle = cola_walkin[[
    'reservation_datetime', 'created_datetime', 'antelacion_horas',
    'status', 'shift', 'people', 'table'
]].sort_values('antelacion_horas')

print("Casos con mayor desfase (los 15 más extremos):")
print(cola_walkin_detalle.head(15).to_string())
print()

# ¿Se concentran en fechas concretas? Si varias filas comparten la misma
# reservation_date, apunta a un problema del día (ej. caída del sistema,
# entrada manual retrasada por lote) más que a errores individuales sueltos.
print("Reservas por fecha dentro de la cola (¿se repite alguna fecha?):")
print(cola_walkin_detalle['reservation_datetime'].dt.date.value_counts().head(10))

Walk-in con antelación < -2h: 1 de 3063 (0.03%)

Distribución de antelación dentro de la cola:
count     1.000000
mean    -22.193889
std            NaN
min     -22.193889
25%     -22.193889
50%     -22.193889
75%     -22.193889
max     -22.193889
Name: antelacion_horas, dtype: float64

Casos con mayor desfase (los 15 más extremos):
      reservation_datetime    created_datetime  antelacion_horas   status   shift  people table
16431  2025-09-24 13:00:00 2025-09-25 11:11:38        -22.193889  Sentada  Comida       1    10

Reservas por fecha dentro de la cola (¿se repite alguna fecha?):
reservation_datetime
2025-09-24    1
Name: count, dtype: int64


In [9]:
import pandas as pd

# ---------------------------------------------------------------------------
# Hipótesis de Ana, sin mezclar origin y status: dentro de TODAS las
# reservas con status == 'Liberada' (cualquier canal), ¿hay una proporción
# relevante con antelación casi nula (creada minutos antes del servicio)?
# ---------------------------------------------------------------------------

liberada_todas = df_reservas[df_reservas['status'] == 'Liberada'].copy()

print(f"Total 'Liberada' (todos los canales): {len(liberada_todas)}\n")

print("Distribución de antelación (horas) en Liberada — todos los canales:")
print(liberada_todas['antelacion_horas'].describe())
print()

for umbral in [0.25, 1, 6, 24]:  # 15min, 1h, 6h, 1 día
    pct = (liberada_todas['antelacion_horas'].abs() < umbral).mean() * 100
    print(f"% Liberada con antelación < {umbral}h: {pct:.1f}%")
print()

# Desglose por canal DENTRO de Liberada: ¿el patrón de antelación corta
# es transversal a todos los orígenes, o se concentra en alguno concreto?
print("Antelación media/mediana de Liberada, desglosado por origin:")
print(liberada_todas.groupby('origin')['antelacion_horas'].agg(['count', 'mean', 'median']))
print()

# Casos concretos con antelación muy corta, para inspección manual,
# como los que probablemente vio Ana
casos_extremos = liberada_todas[
    liberada_todas['antelacion_horas'].abs() < 0.25
][['reservation_datetime', 'created_datetime', 'antelacion_horas', 'origin', 'shift', 'people']]

print(f"Casos con antelación < 15 min: {len(casos_extremos)}")
print(casos_extremos.sort_values('antelacion_horas').to_string())

Total 'Liberada' (todos los canales): 662

Distribución de antelación (horas) en Liberada — todos los canales:
count     662.000000
mean       20.484944
std        68.108005
min        -0.386944
25%        -0.044653
50%         1.853472
75%        18.098472
max      1231.872500
Name: antelacion_horas, dtype: float64

% Liberada con antelación < 0.25h: 27.5%
% Liberada con antelación < 1h: 40.3%
% Liberada con antelación < 6h: 64.8%
% Liberada con antelación < 24h: 81.1%

Antelación media/mediana de Liberada, desglosado por origin:
           count        mean     median
origin                                 
appmovil       9  101.393889  74.090833
moduloweb    121   40.330030  10.939722
software     184   19.344976   2.467361
terceros     165   25.661894   5.087222
walk in      183   -0.137344  -0.135833

Casos con antelación < 15 min: 182
      reservation_datetime    created_datetime  antelacion_horas    origin   shift  people
14257  2025-05-03 13:30:00 2025-05-03 13:44:54         -